# Temperature, top-k and top-p sampling with PRNG keys

Write the token sampler used in a JAX decode loop. Everything must be array ops (no Python loops over the batch or vocabulary), so it can run under `jit`.

Write `sample_tokens(logits, seed, temperature, top_k, top_p)`:

- `logits`: `(B, V)` float32. `temperature > 0`. `top_k`: int, `0` means no top-k filter. `top_p`: float in `(0, 1]`, `1.0` means no top-p filter.

Steps, per row:

1. Divide the logits by `temperature`.
2. **Top-k** (if `top_k > 0`): keep every token whose scaled logit is `>=` the `top_k`-th largest. Ties at the boundary are all kept.
3. **Top-p** (if `top_p < 1`): take the softmax over the tokens still kept. Sort them by probability, descending; equal probabilities are ordered by **lower token id first**. Keep a token if the total probability of the tokens strictly before it in that order is `< top_p`. (So the top token is always kept.)
4. The filtered log-probabilities are the `log_softmax` over the kept tokens, with `-inf` for removed tokens.
5. Sample with exactly `jax.random.categorical(jax.random.PRNGKey(seed), filtered, axis=-1)`.

Return `(tokens, filtered)`: `(B,)` int32 and `(B, V)` float32.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def sample_tokens(logits, seed, temperature, top_k, top_p):
    """Return (tokens (B,), filtered log-probs (B, V) with -inf for removed tokens)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[2.0, 1.0, 0.0, -1.0]], "dtype": "float64"}, 0, 1.0, 2, 1.0], "e": {"__tup__": [{"__nd__": [0], "dtype": "int32"}, {"__nd__": [[-0.31326162815093994, -1.31326162815094, {"__f__": "-inf"}, {"__f__": "-inf"}]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.6931471805599453, -1.2039728043259361, -1.6094379124341003]], "dtype": "float64"}, 1, 1.0, 0, 0.6], "e": {"__tup__": [{"__nd__": [0], "dtype": "int32"}, {"__nd__": [[-0.4700036644935608, -0.9808292984962463, {"__f__": "-inf"}]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.2544482530485257, 0.7769020178860887, 0.9648811301409165, -1.08217838898024, 0.8670173581146392, -0.16816075913655737, -0.1896609791259655, 1.1205664556072035, -0.4367790637355302, -1.2796453624658768], [-0.46949977426329975, -1.1968271623643079, -1.8398571423973407, -0.14555696985382466, 0.398278973208401, -2.1620612747589183, 0.014477306355424725, 0.25519811571103884, 0.173191040730858, -1.2700935204100632], [-0.3109613280418142, -0.1862156647171206, 0.43297608368498736, -1.2873513250071738, 0.38341276871467855, 0.9918800333029093, -2.287645387459913, 0.22393912767437332, -1.5266134112032352, -0.17046692823269938], [-0.6801483671467892, 1.2736350667369072, 0.057108684071811494, 0.7952886919432494, -1.0467789255340256, 0.460527270708541, 1.4223727356191476, -0.7072569194337984, 0.6658737678702557, 0.4813697034349217]], "dtype": "float64"}, 2, 0.7, 5, 0.9], "e": {"__tup__": [{"__nd__": [1, 4, 5, 6], "dtype": "int32"}, {"__nd__": [[-1.2718896865844727, -1.9540987014770508, -1.6855571269989014, {"__f__": "-inf"}, -1.8253625631332397, {"__f__": "-inf"}, {"__f__": "-inf"}, -1.4631494283676147, {"__f__": "-inf"}, {"__f__": "-inf"}], [{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -2.051690101623535, -1.2747817039489746, {"__f__": "-inf"}, -1.8230698108673096, -1.4791829586029053, -1.5963358879089355, {"__f__": "-inf"}], [{"__f__": "-inf"}, {"__f__": "-inf"}, -1.5883246660232544, {"__f__": "-inf"}, -1.6591293811798096, -0.7898904085159302, {"__f__": "-inf"}, -1.8869489431381226, {"__f__": "-inf"}, {"__f__": "-inf"}], [{"__f__": "-inf"}, -1.1510021686553955, {"__f__": "-inf"}, -1.834354043006897, {"__f__": "-inf"}, {"__f__": "-inf"}, -0.9385198354721069, {"__f__": "-inf"}, -2.0192325115203857, {"__f__": "-inf"}]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0, 1.0, 1.0, 0.0], [0.0, 3.0, 3.0, 3.0]], "dtype": "float64"}, 3, 1.0, 2, 1.0], "e": {"__tup__": [{"__nd__": [1, 1], "dtype": "int32"}, {"__nd__": [[-1.0986123085021973, -1.0986123085021973, -1.0986123085021973, {"__f__": "-inf"}], [{"__f__": "-inf"}, -1.0986123085021973, -1.0986123085021973, -1.0986123085021973]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, 4, 1.0, 0, 0.5], "e": {"__tup__": [{"__nd__": [0], "dtype": "int32"}, {"__nd__": [[-0.6931471824645996, -0.6931471824645996, {"__f__": "-inf"}, {"__f__": "-inf"}]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.19402192420958037, -3.7544802690294796, 0.8493059284803959, -0.8108640205376154, 1.8060591168768454, 2.3366472871892365, 4.603620279951209, -2.5096699931411166, 2.5951255781661278, 2.1024606797242207, 0.22626965890897485, 1.835377835131463, 0.3950790611228944, 2.013916969571125, 1.1760012865542868, -1.6878690640411054, 0.3204422767801095, 1.9932901643043344, -2.5546154652435993, -1.1948350886707275, 3.9440654792454453, 0.9744200380654986, -1.8558387881612468, -1.6833860823905902, 4.263042969242991, 1.415919319319834, -5.664553258890576, 1.5792088653938579, -2.6289085490415256, -0.30976657907408156, -4.754422940711187, -0.8164888347425445, -2.267230374404188, -3.589191410480714, 2.7660826161666296, 7.447564891893032, 1.696936582266353, 0.5959668495425247, -3.681814165976925, -0.7595548631392702, -2.006190120958805, -5.601556718198409, 1.651579062547408, -0.1765351385375464, -2.1787065261068608, -4.490012744989938, 1.3334289125639562, 4.049084227858672, 0.28531126503357807, -1.5298545347601746], [1.8389510997382437, -2.9338018425438337, -6.5416259261213465, 5.135785912680596, 5.06223196019619, -0.906177962825242, 0.20506460640757712, 5.00203300727309, -3.3038862901693262, 0.35939951583464597, 2.5984617632112883, 5.214580535966287, -2.4108833631264095, -2.446371146945047, 0.5665970663271459, -1.7397347077403926, 4.63898100486216, -2.2527167968494473, 0.5623891553116639, 0.5214943079596823, -0.8159844551544403, -5.05058668708341, 3.9684646060069735, 0.5922970773797865, 2.0534988898311433, 1.776775425488231, -1.6885413326843368, -1.3056999537682537, -1.3182221390862205, 3.465368784995505, -1.5722004149048443, -6.74844082938902, -0.9550036831698103, -2.759187557760911, 2.786815899238487, -1.9742789266394762, 1.1855418047640287, -0.18795293360426618, 0.5022392748198846, -2.7081490812224516, -1.4179110991847563, -0.24255168448700642, 1.3204263659335473, 1.1522483865741413, -3.5595211557905224, -0.9988111090071018, 2.8474540282532956, 2.0384812369191456, -1.704099454335849, -1.550503840796066], [4.611842333001135, 2.1036914159951836, -5.154006113568764, 6.231931676401079, 2.0507585808474658, 0.7901649477082697, -1.4982726622682696, -3.29650608401505, -1.961598551555949, -2.400067994794404, -3.5232217837350044, -6.148968827921731, 4.143062228470692, 3.958993633390272, 1.7094073833042858, 0.349344113348471, -1.919502248710113, -1.4257731133608367, -0.1827319984227857, -2.687006838394318, 0.8376736728021943, -0.17304785148913382, -1.0125610448242621, 0.5195744198699, 0.17390686915808623, 2.8690314110845203, -0.7941405986994617, -2.9606074730084466, 0.9424351729359636, 3.872294002050179, 5.5093630327197936, 2.7527000042057197, -3.740552380747376, -2.1631633433506186, -4.516972966619736, 0.3773838069249957, -0.455666030287075, -2.955412879930207, 4.812980724658422, 2.7900043101073826, -1.0364817979946859, 2.758495664257609, -5.975915604894663, 1.5573039843257281, 0.6960694736684593, 6.93525168067916, 4.228438972064726, 2.177549359252712, -0.6165194870143859, -0.9728366344222632]], "dtype": "float64"}, 5, 1.5, 0, 0.8], "e": {"__tup__": [{"__nd__": [20, 46, 45], "dtype": "int32"}, {"__nd__": [[{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -3.869097948074341, -2.3577828407287598, {"__f__": "-inf"}, -3.6967790126800537, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -2.797485828399658, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -2.584834337234497, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -3.582807779312134, -0.4618193805217743, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -2.727473497390747, {"__f__": "-inf"}, {"__f__": "-inf"}], [{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -1.7979146242141724, -1.846950650215149, {"__f__": "-inf"}, {"__f__": "-inf"}, -1.8870831727981567, {"__f__": "-inf"}, {"__f__": "-inf"}, -3.489464282989502, -1.7453850507736206, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -2.129117965698242, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -2.5761289596557617, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -2.9115262031555176, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -3.3638949394226074, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -3.3234691619873047, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}], [-2.6656906604766846, {"__f__": "-inf"}, {"__f__": "-inf"}, -1.5856311321258545, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -2.978210687637329, -3.1009230613708496, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -3.1587228775024414, -2.0673434734344482, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -2.5315983295440674, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -1.116750955581665, -2.921293020248413, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.45144139940017075, -0.0836009522433776, -0.5653535701578021, 1.398150669780622, -0.3642549289527679, -1.639190713356673, 0.4598454039730155, -0.4506184308475007, 0.26123633081725056, 0.9471411214505782, -1.1803793769350224, -1.7762522006137178, -0.9730332045850149, 0.8108094488019384, 1.8040451692833035, 1.6043416053354285, -0.3931938142392006, -0.008952318148844384, 0.43268433563433606, -0.7501915519751885], [0.021205253163433684, -0.17070496047256015, 1.1420031574823524, -0.4266719285261043, 1.8921456173826185, 0.855148221604705, 0.16997692375600676, -1.6159720026513442, 0.21602935870258502, 0.09585252816545163, 1.4054502635147008, 0.3909887102379216, 2.2235998926779232, -0.06647103040418631, -1.634151055831145, 0.585520038391888, -1.3250569549224043, 0.12992417087433508, -0.6834013616924528, 1.1920863006075069]], "dtype": "float64"}, 6, 0.1, 0, 1.0], "e": {"__tup__": [{"__nd__": [14, 12], "dtype": "int32"}, {"__nd__": [[-13.668619155883789, -19.01904296875, -23.83656883239746, -4.201526165008545, -21.82558250427246, -34.5749397277832, -13.584579467773438, -22.68921661376953, -15.570669174194336, -8.71162223815918, -29.986827850341797, -35.945552825927734, -27.91336441040039, -10.074938774108887, -0.14258155226707458, -2.139615535736084, -22.114971160888672, -18.27255630493164, -13.856189727783203, -25.684947967529297], [-22.059974670410156, -23.97907829284668, -10.851996421813965, -26.538747787475586, -3.3505725860595703, -13.720545768737793, -20.57225799560547, -38.4317512512207, -20.11173439025879, -21.31350326538086, -8.217525482177734, -18.362140655517578, -0.03602799400687218, -22.936738967895508, -38.61354064941406, -16.416828155517578, -35.52259826660156, -20.97278594970703, -29.106040954589844, -10.351164817810059]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.8535479911174263, 0.8170308970266977, -1.6994326826169839, 1.0424517582385187, -1.1119134110821722, 0.3325913957989379, 1.128897656723177, -0.43703410693047606], [-0.08139632202485023, 1.3314927367213372, 0.18487490982632518, 0.13374077053196287, 1.3492986187904532, 0.5483208969143776, -0.04146248688277244, 0.40941700995598407], [-2.1196984305972273, -2.60718327764613, -0.9596171462067122, 0.5364402560142466, -0.8934931578523152, -0.5456494823449283, 0.3027899550128016, 0.7196014818061905], [0.6562243672541257, -0.553179271502744, 0.5409472886384353, -0.15880066995562558, 0.15866792178888792, 1.2185612397962056, 2.578441350578373, -1.7582257635957796], [-0.6348686701817856, -1.5580541070020986, -1.0904164250613737, 1.3375075077796337, 2.7139932026137803, 1.06167033009449, -0.538931857479624, -1.59592110920454]], "dtype": "float64"}, 7, 1.0, 1, 0.3], "e": {"__tup__": [{"__nd__": [6, 4, 7, 6, 4], "dtype": "int32"}, {"__nd__": [[{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, 0.0, {"__f__": "-inf"}], [{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, 0.0, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}], [{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, 0.0], [{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, 0.0, {"__f__": "-inf"}], [{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, 0.0, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(sample_tokens, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")